In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
from datetime import datetime
from sklearn.model_selection import train_test_split, learning_curve
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.tree import DecisionTreeClassifier

df = pd.read_csv('df.csv')
if 'Unnamed: 0' in df.columns:
    df.drop('Unnamed: 0', axis=1, inplace=True)

In [ ]:
patterns = [
    "draft",
    "account_id_",
    "party_id",
    "hero_variant",
    "name_",
    "isRadiant_",
    "rank_tier_",
    "game_mode",
    "lobby_type",
    "start_time",
"lane_", "is_roaming",
    "version",
    "series_type",
    "patch",
    "region",
    "radiant_win"
]

# Фильтруем столбцы: оставляем только те, в имени которых встречается хотя бы одна из подстрок
cols_to_keep = [col for col in df.columns if any(pattern in col for pattern in patterns)]

# Создаем новый DataFrame только с выбранными столбцами
df = df[cols_to_keep]

# Определяем X и y (предполагается, что целевая переменная называется 'radiant_win')
if 'radiant_win' not in df.columns:
    raise ValueError("В датасете отсутствует столбец 'radiant_win'")
X = df.drop(columns=['radiant_win'])
y = df['radiant_win']

# Разбиваем данные на обучающую и тестовую выборки
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

In [ ]:
# Определяем числовые и категориальные признаки
numeric_features = X_train.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=[np.number]).columns.tolist()

# Создаем пайплайн для числовых признаков: сначала заполняем пропуски, потом масштабируем
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

# Для категориальных признаков: заполняем пропуски пустой строкой, затем применяем OneHotEncoder
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Объединяем трансформации
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

In [ ]:
# Функция для проведения эксперимента с моделью
def run_experiment(model_name, model_pipeline, experiment_id):
    # Вычисляем learning curve на обучающей выборке
    # Сначала обрабатываем X_train через preprocessor
    X_train_processed = model_pipeline.named_steps['preprocessor'].fit_transform(X_train)
    train_sizes, train_scores, val_scores = learning_curve(
        model_pipeline.named_steps['classifier'], X_train_processed, y_train,
        cv=3, train_sizes=np.linspace(0.1, 1.0, 10)
    )
    train_scores_mean = np.mean(train_scores, axis=1)
    val_scores_mean = np.mean(val_scores, axis=1)
    
    # Обучаем пайплайн на обучающих данных
    model_pipeline.fit(X_train, y_train)
    
    # Предсказания на тестовой выборке
    y_pred = model_pipeline.predict(X_test)
    
    # Вычисляем метрики
    acc = accuracy_score(y_test, y_pred)
    conf_matrix = confusion_matrix(y_test, y_pred).tolist()
    class_report = classification_report(y_test, y_pred, output_dict=True)
    
    experiment = {
        "model": model_name,
        "params": model_pipeline.named_steps['classifier'].get_params(),
        "learning_curve": {
            "train_sizes": train_sizes.tolist(),
            "train_scores": train_scores_mean.tolist(),
            "val_scores": val_scores_mean.tolist(),
        },
        "test_score": acc,
        "confusion_matrix": conf_matrix,
        "classification_report": class_report,
        "timestamp": datetime.now().isoformat(),
        "experiment_id": experiment_id
    }
    
    # Сохраняем эксперимент в pickle-файл
    if not os.path.exists("experiments"):
        os.makedirs("experiments")
    filename = os.path.join("experiments", f"{model_name}_experiment_{experiment_id}.pkl")
    with open(filename, "wb") as f:
        pickle.dump(experiment, f)
    print(f"Сохранен эксперимент: {filename}")

experiment_counter = 1

# Задаем список моделей и для каждой два набора гиперпараметров
models_experiments = [
    ("LogisticRegression", LogisticRegression, [
        {"C": 1.0, "penalty": "l2", "solver": "lbfgs", "max_iter": 200},
        {"C": 0.8, "penalty": "l2", "solver": "lbfgs", "max_iter": 200}
    ]),
    ("SGDClassifier", SGDClassifier, [
        {"loss": "hinge", "max_iter": 1000, "tol": 1e-3},
        {"loss": "log", "max_iter": 1000, "tol": 1e-3}
    ]),
    ("DecisionTreeClassifier", DecisionTreeClassifier, [
        {"max_depth": 5, "criterion": "gini"},
        {"max_depth": 7, "criterion": "gini"}
    ])
]

# Для каждой модели проводим два эксперимента
for model_name, model_class, params_list in models_experiments:
    for params in params_list:
        # Создаем пайплайн с предобработкой и выбранной моделью с гиперпараметрами
        classifier = model_class(**params)
        pipeline = Pipeline(steps=[
            ('preprocessor', preprocessor),
            ('classifier', classifier)
        ])
        run_experiment(model_name, pipeline, experiment_counter)
        experiment_counter += 1

print("Все эксперименты завершены.")